In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from features import FeatureEngineer
from modelmanager import ModelManager
from evaluator import Evaluator

fe = FeatureEngineer()
mm = ModelManager()
ev = Evaluator()

In [ ]:
demo_raw = pd.read_csv("../data/BlackFriday-Demographic.csv")
click_raw = pd.read_csv("../data/Click_Button.csv")
purchase_raw = pd.read_csv("../data/Purchases.csv")

print(demo_raw.shape)
print(click_raw.shape)
print(purchase_raw.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'BlackFriday-Demographic.csv'

In [ ]:
demo_clean = fe.clean_demographic(demo_raw)
click_clean = fe.clean_clickstream(click_raw)
purchase_clean = fe.clean_purchases(purchase_raw)
purchase_agg = fe.aggregate_purchases(purchase_clean)

demo_encoded = fe.encode(demo_clean)
click_encoded = fe.encode(click_clean)
purchase_encoded = fe.encode(purchase_agg)

print(demo_encoded.shape)
print(click_encoded.shape)
print(purchase_encoded.shape)

In [ ]:
demo_scaled, demo_scaler = mm.scale_data(demo_encoded)
click_scaled, click_scaler = mm.scale_data(click_encoded)
purchase_scaled, purchase_scaler = mm.scale_data(purchase_encoded)

demo_pca, demo_pca_model = mm.apply_pca(demo_scaled)
click_pca, click_pca_model = mm.apply_pca(click_scaled)
purchase_pca, purchase_pca_model = mm.apply_pca(purchase_scaled)

print(demo_pca.shape)
print(click_pca.shape)
print(purchase_pca.shape)

In [ ]:
demo_gmm, demo_labels, demo_probs, demo_scores = mm.choose_best_gmm(demo_pca)
click_gmm, click_labels, click_probs, click_scores = mm.choose_best_gmm(click_pca)
purchase_gmm, purchase_labels, purchase_probs, purchase_scores = mm.choose_best_gmm(purchase_pca)

print("Demographic GMM scores")
display(demo_scores)

print("Clickstream GMM scores")
display(click_scores)

print("Purchase GMM scores")
display(purchase_scores)

In [ ]:
demo_scores.plot(x="k", y="bic", marker="o", title="Demographic BIC")
plt.show()

click_scores.plot(x="k", y="bic", marker="o", title="Clickstream BIC")
plt.show()

purchase_scores.plot(x="k", y="bic", marker="o", title="Purchase BIC")
plt.show()

In [ ]:
demo_profile, demo_sizes = ev.cluster_profile(demo_encoded, demo_labels, "Demographic")
click_profile, click_sizes = ev.cluster_profile(click_encoded, click_labels, "Clickstream")
purchase_profile, purchase_sizes = ev.cluster_profile(purchase_encoded, purchase_labels, "Purchase")

In [ ]:
demo_top = ev.top_cluster_features(demo_profile)
click_top = ev.top_cluster_features(click_profile)
purchase_top = ev.top_cluster_features(purchase_profile)

print("Demographic top features")
display(demo_top)

print("Clickstream top features")
display(click_top)

print("Purchase top features")
display(purchase_top)

In [ ]:
comparison, corr_matrix = ev.compare_datasets(
    demo_profile,
    click_profile,
    purchase_profile
)

display(comparison)
display(corr_matrix)

In [ ]:
demo_output = demo_clean.copy()
demo_output["cluster"] = demo_labels

click_output = click_clean.copy()
click_output["cluster"] = click_labels

purchase_output = purchase_agg.copy()
purchase_output["cluster"] = purchase_labels

demo_output.to_csv("demographic_clustered.csv", index=False)
click_output.to_csv("clickstream_clustered.csv", index=False)
purchase_output.to_csv("purchase_clustered.csv", index=False)

comparison.to_csv("segment_level_comparison.csv")
corr_matrix.to_csv("segment_level_correlations.csv")